# Phase 1.2 — Indexing & Views

**Critical distinction:** some operations share memory (views), others copy.

Getting this wrong silently corrupts data during training.

In [ ]:
import torch

t = torch.arange(12).reshape(3, 4)
print(t)

## Indexing (like NumPy)

```python
t[row, col]     # single element
t[0]            # first row
t[:, 1]         # second column
t[0:2, 1:3]     # slice rows & cols
```

In [ ]:
print("element t[1,2]:", t[1, 2].item())
print("row 0:\n", t[0])
print("col 1:\n", t[:, 1])
print("submatrix:\n", t[0:2, 1:3])

## View vs copy

| Operation | Shares memory? |
|-----------|----------------|
| `t.view(...)` / `t.reshape(...)` | Usually yes (view) |
| `t[0]` / slicing | Yes (view) |
| `t.clone()` | No (copy) |
| `t.contiguous()` | May copy if layout is non-contiguous |

**Test:** modify a view → original changes too.

In [ ]:
original = torch.tensor([1., 2., 3., 4.])
view = original.view(2, 2)

view[0, 0] = 99
print("original after editing view:", original)  # [99, 2, 3, 4] — shared!

safe = original.clone()
safe[0] = -1
print("original after editing clone:", original)  # unchanged

## `.T` transpose — the gotcha

Transposing can make a tensor **non-contiguous**. Then `.view()` fails; use `.reshape()` or `.contiguous().view()`.

In [ ]:
x = torch.arange(6).reshape(2, 3)
xt = x.T

print(f"x contiguous?  {x.is_contiguous()}")
print(f"x.T contiguous? {xt.is_contiguous()}")

# xt.view(-1)       # RuntimeError!
xt.reshape(-1)      # works — handles non-contiguous
print("reshape OK:", xt.reshape(-1).tolist())

## ✏️ Exercise

Given `m` below:
1. Extract **column 2** into `col2` (shape `(3,)`)
2. Set **row 1** of `m` to all zeros **in-place** via slicing
3. Create `flat` = flattened copy (not a view) of the modified `m`

In [ ]:
m = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print("m:\n", m)

# YOUR CODE
col2 = ...
# zero row 1 in-place ...
flat = ...

print("\ncol2:", col2)
print("m after zero row:\n", m)
print("flat:", flat)

In [ ]:
assert col2.shape == (3,)
assert torch.equal(col2, torch.tensor([2., 6., 10.]))
assert torch.all(m[1] == 0)
assert flat.shape == (12,)
assert not flat.data_ptr() == m.data_ptr()  # must be a copy
print("✓ Phase 1.2 passed!")